# RAG with BERTopic Topic-Based Chunking

this pipeline uses **BERTopic** to group sentences into semantically coherent topic clusters. Each cluster becomes one chunk — meaning the retriever always fetches topically unified context rather than arbitrary text windows.

## Installations

In [ ]:
import subprocess
import time
import warnings
warnings.filterwarnings("ignore")

!sudo apt-get install -y lshw
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print("Starting server...")
time.sleep(5)

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  pci.ids usb.ids
The following NEW packages will be installed:
  lshw pci.ids usb.ids
0 upgraded, 3 newly installed, 0 to remove and 42 not upgraded.
Need to get 791 kB of archives.
After this operation, 2,988 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 lshw amd64 02.19.git.2021.06.19.996aaad9c7-2build1 [321 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 usb.ids all 2022.04.02-1 [219 kB]
Fetched 791 kB in 1s (627 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 3.)
debconf: falling back to frontend:

In [ ]:
!ollama pull mistral
!ollama list


NAME              ID              SIZE      MODIFIED               
mistral:latest    6577803aa9a0    4.4 GB    Less than a second ago    


In [ ]:
%pip install pypdf
%pip install faiss-gpu-cu12
%pip install -U langchain-community
%pip install langchain-ollama
%pip install colorama

# BERTopic and its core dependencies
%pip install bertopic
%pip install sentence-transformers   # for embedding sentences before topic modelling
%pip install umap-learn              # dimensionality reduction used by BERTopic
%pip install hdbscan                 # clustering algorithm used by BERTopic
%pip install datasets                # HuggingFace datasets (for load_dataset support)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.5/334.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.9 MB/s eta 0:00:00


In [ ]:
!pip install cudf-cu12 cuml-cu12 --extra-index-url https://pypi.nvidia.com

Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 MB 115.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-cuda-nvcc-cu12
    Found existing installation: nvidia-cuda-nvcc-cu12 12.5.82
    Uninstalling nvidia-cuda-nvcc-cu12-12.5.82:
      Successfully uninstalled nvidia-cuda-nvcc-cu12-12.5.82


## Imports

In [ ]:
import re
import time
import threading
import numpy as np
from typing import List, Optional, Union
from tqdm.auto import tqdm
from colorama import Fore
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from langchain_core.documents import Document
from langchain_ollama import ChatOllama
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import (
    PyPDFLoader,
    CSVLoader,
    Docx2txtLoader,
    UnstructuredExcelLoader,
)
from langchain_community.vectorstores import FAISS
from datasets import Dataset as HFDataset

try:
    import torch
    from cuml.manifold import UMAP as cuUMAP
    from cuml.cluster import HDBSCAN as cuHDBSCAN
    CUML_AVAILABLE = torch.cuda.is_available()
except ImportError:
    CUML_AVAILABLE = False


## BERTopic Chunker

It works in three steps:

1. **Sentence segmentation** — split raw text into individual sentences.
2. **Topic modelling** — embed every sentence with a `SentenceTransformer`, then run BERTopic (UMAP + HDBSCAN + c-TF-IDF) to assign each sentence a topic ID.
3. **Chunk assembly** — concatenate consecutive sentences that share the same topic ID into a single chunk.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Timed BERTopic fit
# ══════════════════════════════════════════════════════════════════════════════

def _timed_fit_transform(topic_model, all_sentences, embeddings):
    """
    Run topic_model.fit_transform() in a background thread while printing
    a live elapsed-time counter. BERTopic exposes no progress hooks so
    elapsed time + estimated stage is the best feedback available.
    """
    result = [None]
    error  = [None]

    def _run():
        try:
            result[0] = topic_model.fit_transform(
                all_sentences, embeddings=embeddings,
            )
        except Exception as e:
            error[0] = e

    thread = threading.Thread(target=_run, daemon=True)
    thread.start()

    stages           = ["UMAP", "HDBSCAN", "c-TF-IDF"]
    stage_thresholds = [0.60,    0.85,      1.01]
    cap              = 1800
    start            = time.time()

    while thread.is_alive():
        elapsed    = time.time() - start
        frac       = min(elapsed / cap, 0.99)
        stage      = next(
            (s for s, t in zip(stages, stage_thresholds) if frac < t),
            stages[-1],
        )
        filled     = int(frac * 30)
        bar        = "█" * filled + "░" * (30 - filled)
        mins, secs = divmod(int(elapsed), 60)
        print(f"\r🧠 [{bar}] {mins:02d}:{secs:02d}  stage: {stage}   ",
              end="", flush=True)
        time.sleep(1)

    thread.join()
    elapsed    = time.time() - start
    mins, secs = divmod(int(elapsed), 60)
    print(f"\r🧠 [{'█' * 30}] {mins:02d}:{secs:02d}  ✔ Done!                    ")

    if error[0]:
        raise error[0]
    return result[0]


# ══════════════════════════════════════════════════════════════════════════════
# BERTopicChunker
# ══════════════════════════════════════════════════════════════════════════════

class BERTopicChunker:
    """
    Topic-aware chunker powered by BERTopic.

    • BERTopic is fit ONCE across the entire corpus  →  avoids k >= N UMAP crash.
    • UMAP + HDBSCAN use cuML GPU backends when available, CPU fallback otherwise.
    • Chunk assembly is fully vectorized with NumPy  →  no per-document Python loop.

    Parameters
    ----------
    embedding_model : str
        SentenceTransformer model for encoding sentences before topic modelling.
    min_topic_size : int
        Minimum sentences per topic cluster. Use 100-200 for large corpora to
        avoid thousands of micro-topics that bloat FAISS indexing.
    min_chunk_sentences : int
        Minimum sentences per assembled chunk. Short tails are folded into
        the previous chunk rather than discarded.
    umap_n_neighbors : int
        UMAP n_neighbors — must be < total sentence count in corpus.
    umap_n_components : int
        UMAP output dimensionality fed into HDBSCAN.
    use_gpu : bool
        Use cuML GPU-accelerated UMAP + HDBSCAN when available.
    """

    def __init__(
        self,
        embedding_model: str     = "paraphrase-multilingual-MiniLM-L12-v2",
        min_topic_size: int      = 150,
        min_chunk_sentences: int = 2,
        umap_n_neighbors: int    = 15,
        umap_n_components: int   = 5,
        use_gpu: bool            = True,
    ):
        self.sentence_model      = SentenceTransformer(embedding_model)
        self.min_topic_size      = min_topic_size
        self.min_chunk_sentences = min_chunk_sentences
        self._is_fitted          = False

        gpu = use_gpu and CUML_AVAILABLE
        if gpu:
            print(Fore.GREEN + "  ✔ GPU detected — using cuML UMAP + HDBSCAN.")
            self.umap_model = cuUMAP(
                n_neighbors  = umap_n_neighbors,
                n_components = umap_n_components,
                min_dist     = 0.0,
                metric       = "cosine",
            )
            self.hdbscan_model = cuHDBSCAN(
                min_cluster_size         = self.min_topic_size,
                metric                   = "euclidean",
                cluster_selection_method = "eom",
                prediction_data          = True,
            )
        else:
            print(Fore.YELLOW + "  ⚠ No GPU / cuML — using CPU UMAP + HDBSCAN.")
            self.umap_model = UMAP(
                n_neighbors  = umap_n_neighbors,
                n_components = umap_n_components,
                min_dist     = 0.0,
                metric       = "cosine",
                random_state = 42,
            )
            self.hdbscan_model = HDBSCAN(
                min_cluster_size         = self.min_topic_size,
                metric                   = "euclidean",
                cluster_selection_method = "eom",
                prediction_data          = True,
            )

        self.topic_model = BERTopic(
            embedding_model = self.sentence_model,
            umap_model      = self.umap_model,
            hdbscan_model   = self.hdbscan_model,
            verbose         = True,
        )

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------

    @staticmethod
    def _split_sentences(text: str) -> List[str]:
        """Regex sentence splitter — drops fragments shorter than 20 chars."""
        sentences = re.split(r'(?<=[.!?])\s+', text.strip())
        return [s.strip() for s in sentences if len(s.strip()) > 20]

    @staticmethod
    def _assemble_chunks_vectorized(
        all_sentences: List[str],
        topics: List[int],
        doc_sentence_spans: List[tuple],
        documents: List[Document],
        min_chunk_sentences: int,
    ) -> List[Document]:
        """
        Vectorized chunk assembly over the entire corpus in one pass.

        1. Build a NumPy topic + doc-index array for all sentences at once.
        2. Compute boundary positions where topic OR document changes.
        3. Slice all_sentences at those positions in one pass.
        4. Attach per-document metadata to each resulting chunk.
        """
        n = len(all_sentences)
        if n == 0:
            return []

        topics_arr = np.array(topics, dtype=np.int32)

        doc_idx = np.empty(n, dtype=np.int32)
        for d_i, (start, end) in enumerate(doc_sentence_spans):
            doc_idx[start:end] = d_i

        topic_change     = np.empty(n, dtype=bool)
        topic_change[0]  = True
        topic_change[1:] = (
            (topics_arr[1:] != topics_arr[:-1]) &
            (topics_arr[1:] != -1)
        )
        doc_change     = np.empty(n, dtype=bool)
        doc_change[0]  = False
        doc_change[1:] = doc_idx[1:] != doc_idx[:-1]

        boundary_positions = np.where(topic_change | doc_change)[0].tolist() + [n]

        all_sentences_arr = np.array(all_sentences, dtype=object)
        raw_chunks: List[tuple] = []
        chunk_counter = np.zeros(len(documents), dtype=np.int32)

        prev = 0
        for pos in boundary_positions[1:]:
            slice_sentences = all_sentences_arr[prev:pos].tolist()
            d_i = int(doc_idx[prev])

            if len(slice_sentences) < min_chunk_sentences:
                if raw_chunks and raw_chunks[-1][0] == d_i:
                    prev_d, prev_ci, prev_text = raw_chunks[-1]
                    raw_chunks[-1] = (prev_d, prev_ci,
                                      prev_text + " " + " ".join(slice_sentences))
                else:
                    raw_chunks.append((d_i, int(chunk_counter[d_i]),
                                       " ".join(slice_sentences)))
                    chunk_counter[d_i] += 1
            else:
                raw_chunks.append((d_i, int(chunk_counter[d_i]),
                                   " ".join(slice_sentences)))
                chunk_counter[d_i] += 1

            prev = pos

        return [
            Document(
                page_content=text,
                metadata={**documents[d_i].metadata, "chunk_index": c_i},
            )
            for d_i, c_i, text in raw_chunks
        ]

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def split_documents(self, documents: List[Document]) -> List[Document]:
        """
        Fit BERTopic once over all documents, then assemble chunks in a
        single vectorized NumPy pass — no per-document Python loop.
        """
        # Phase 1 — sentence collection
        all_sentences: List[str] = []
        doc_sentence_spans: List[tuple] = []

        with tqdm(documents, desc="📄 Collecting sentences", unit="doc") as pbar:
            for doc in pbar:
                sentences = self._split_sentences(doc.page_content)
                start = len(all_sentences)
                all_sentences.extend(sentences)
                doc_sentence_spans.append((start, len(all_sentences)))
                pbar.set_postfix(sentences=len(all_sentences))

        if not all_sentences:
            return []

        # Phase 2a — sentence embedding
        print(Fore.CYAN + f"  → Embedding {len(all_sentences)} sentences...")
        embeddings = self.sentence_model.encode(
            all_sentences,
            show_progress_bar=True,
            batch_size=256,
        )

        # Phase 2b — UMAP + HDBSCAN + c-TF-IDF
        print(Fore.CYAN + "  → Running UMAP → HDBSCAN → c-TF-IDF...")
        topics, _ = _timed_fit_transform(self.topic_model, all_sentences, embeddings)

        self._is_fitted = True
        n_topics = len(set(t for t in topics if t != -1))
        print(Fore.CYAN + f"  ✔ Found {n_topics} topics across {len(all_sentences)} sentences.")

        # Phase 3 — vectorized chunk assembly
        print(Fore.CYAN + "  → Assembling chunks (vectorized)...")
        chunked = self._assemble_chunks_vectorized(
            all_sentences, topics, doc_sentence_spans,
            documents, self.min_chunk_sentences,
        )
        print(Fore.CYAN + f"  ✔ Produced {len(chunked)} topic-coherent chunks.")
        return chunked


# ══════════════════════════════════════════════════════════════════════════════
# RAG
# ══════════════════════════════════════════════════════════════════════════════

class RAG:

    def __init__(
        self,
        ollama_model: str             = "mistral:instruct",
        bertopic_embedding_model: str = "paraphrase-multilingual-MiniLM-L12-v2",
        min_topic_size: int           = 150,
        umap_n_neighbors: int         = 15,
        top_k: int                    = 3,
        use_gpu: bool                 = True,
        faiss_batch_size: int         = 512,
    ):
        self.top_k            = top_k
        self.faiss_batch_size = faiss_batch_size
        self.vector_db        = None
        self.llm              = ChatOllama(model=ollama_model)

        import torch
        device = "cuda" if (use_gpu and torch.cuda.is_available()) else "cpu"
        print(Fore.GREEN + f"  ✔ FAISS embedder running on {device.upper()}.")

        self.embedder = HuggingFaceEmbeddings(
            model_name    = bertopic_embedding_model,
            model_kwargs  = {"device": device},
            encode_kwargs = {
                "batch_size":        faiss_batch_size,
            },
        )

        self.chunker = BERTopicChunker(
            embedding_model  = bertopic_embedding_model,
            min_topic_size   = min_topic_size,
            umap_n_neighbors = umap_n_neighbors,
            use_gpu          = use_gpu,
        )

    # ------------------------------------------------------------------
    # Document loading
    # ------------------------------------------------------------------

    def load_documents(self, paths: List[str]) -> List[Document]:
        """Load from disk. Supported: .pdf .csv .docx .xlsx"""
        raw_docs: List[Document] = []
        for path in paths:
            try:
                if   path.endswith(".pdf"):  loader = PyPDFLoader(path)
                elif path.endswith(".csv"):  loader = CSVLoader(path)
                elif path.endswith(".docx"): loader = Docx2txtLoader(path)
                elif path.endswith(".xlsx"): loader = UnstructuredExcelLoader(path)
                else:
                    print(Fore.YELLOW + f"Skipping unsupported format: {path}")
                    continue
                raw_docs.extend(loader.load())
            except Exception as e:
                print(Fore.RED + f"Couldn't load {path}: {e}")
        return raw_docs

    def load_from_hf_dataset(
        self,
        dataset: HFDataset,
        text_fields: Union[str, List[str]],
        meta_fields: Optional[List[str]] = None,
        concat_sep: str = "\n",
    ) -> List[Document]:
        if isinstance(text_fields, str):
            text_fields = [text_fields]

        available = set(dataset.column_names)
        missing   = [f for f in text_fields if f not in available]
        if missing:
            raise ValueError(
                f"text_fields {missing} not found.\n"
                f"Available columns: {sorted(available)}"
            )

        meta_fields = [f for f in (meta_fields or []) if f in available]
        raw_docs: List[Document] = []
        skipped = 0

        for i, row in enumerate(dataset):
            parts = [str(row[f]).strip() for f in text_fields if row[f]]
            if not parts:
                skipped += 1
                continue
            metadata = {"hf_row_index": i, "source": "huggingface_dataset"}
            for f in meta_fields:
                metadata[f] = row[f]
            raw_docs.append(Document(
                page_content=concat_sep.join(parts),
                metadata=metadata,
            ))

        print(
            Fore.CYAN
            + f"  → Loaded {len(raw_docs)} rows from HuggingFace dataset"
            + (f" ({skipped} empty rows skipped)." if skipped else ".")
        )
        return raw_docs

    # ------------------------------------------------------------------
    # Chunking + vector store
    # ------------------------------------------------------------------

    def chunk_documents(self, raw_docs: List[Document]) -> List[Document]:
        """Apply BERTopic chunking. Call once — result feeds create_vectorstore()."""
        print(Fore.CYAN + f"  → Chunking {len(raw_docs)} documents with BERTopic...")
        return self.chunker.split_documents(raw_docs)

    def create_vectorstore(self, chunks: List[Document]) -> None:
        """
        Embed chunks with HuggingFaceEmbeddings (local, batched, GPU) and
        build a FAISS index. Call once — then use query() repeatedly.
        """
        if not chunks:
            raise ValueError("No chunks — check documents loaded correctly.")

        print(Fore.CYAN + f"  → Building FAISS index from {len(chunks)} chunks...")
        texts      = [c.page_content for c in chunks]
        metadatas  = [c.metadata     for c in chunks]
        embeddings = self.embedder.embed_documents(texts)

        self.vector_db = FAISS.from_embeddings(
            text_embeddings = list(zip(texts, embeddings)),
            embedding       = self.embedder,
            metadatas       = metadatas,
        )
        print(Fore.CYAN + f"  ✔ FAISS index built with {len(chunks)} chunks.")

    def save_index(self, path: str = "faiss_index") -> None:
        """
        Persist the FAISS index to disk. On the next session call
        load_index() to skip all chunking + embedding entirely.
        """
        if self.vector_db is None:
            raise RuntimeError("No index to save — call create_vectorstore() first.")
        self.vector_db.save_local(path)
        print(Fore.GREEN + f"  ✔ FAISS index saved to '{path}'.")

    def load_index(self, path: str = "faiss_index") -> None:
        """
        Load a previously saved FAISS index from disk.
        Skips all loading, chunking, and embedding.
        """
        self.vector_db = FAISS.load_local(
            path,
            self.embedder,
            allow_dangerous_deserialization=True,
        )
        print(Fore.GREEN + f"  ✔ FAISS index loaded from '{path}'.")

    # ------------------------------------------------------------------
    # Querying  ←  the only method you call repeatedly
    # ------------------------------------------------------------------

    def query(self, question: str) -> str:
        """
        Query the vector store and generate an answer.
        Requires create_vectorstore() or load_index() to have been called first.
        Does NOT re-chunk or re-embed anything.
        """
        if self.vector_db is None:
            raise RuntimeError(
                "Vector store not ready.\n"
                "Run setup first:\n"
                "  raw_docs = rag.load_from_hf_dataset(...)\n"
                "  chunks   = rag.chunk_documents(raw_docs)\n"
                "  rag.create_vectorstore(chunks)\n"
                "Or reload a saved index:\n"
                "  rag.load_index('faiss_index')"
            )

        results = self.vector_db.similarity_search(question, k=self.top_k)
        context = "\n\n".join(
            f"[Chunk {i+1}]\n{doc.page_content}" for i, doc in enumerate(results)
        )
        prompt = (
            "You are a helpful assistant. Answer the question using ONLY "
            "the provided context. If the context does not contain enough "
            "information, say so clearly.\n\n"
            f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
        )
        return self.llm.invoke(prompt).content

    # ------------------------------------------------------------------
    # analyze() kept for backward compatibility with file-based workflows
    # ------------------------------------------------------------------

    def analyze(
        self,
        query: str,
        file_paths: Optional[List[str]]      = None,
        raw_docs:   Optional[List[Document]] = None,
    ) -> str:
        """
        One-shot pipeline for quick use with small corpora.
        For large datasets use the explicit setup + query() pattern instead.
        """
        if file_paths is None and raw_docs is None:
            raise ValueError("Provide either file_paths or raw_docs.")
        if file_paths is not None and raw_docs is not None:
            raise ValueError("Provide file_paths OR raw_docs, not both.")

        if file_paths is not None:
            raw_docs = self.load_documents(file_paths)

        if not raw_docs:
            return "No valid documents could be loaded."

        chunks = self.chunk_documents(raw_docs)
        if not chunks:
            return "BERTopic produced no chunks — documents may be too short."

        self.create_vectorstore(chunks)
        return self.query(query)

## RAG Pipeline

## Usage — local files

In [ ]:
analyzer = RAG(
    ollama_model="mistral",
    bertopic_embedding_model="all-MiniLM-L6-v2",  # fast, CPU-friendly
    min_topic_size=3,   # lower → finer-grained topics
    top_k=3,            # chunks retrieved per query
)

# Query 1
result1 = analyzer.analyze(
    query="tell me about python",
    file_paths=["java.pdf", "python.pdf", "rust.pdf"],
)
print(result1, "\n")

# Query 2
result2 = analyzer.analyze(
    query="what is the difference between python, java and rust?",
    file_paths=["java.pdf", "python.pdf", "rust.pdf"],
)
print(result2)

## Usage — HuggingFace dataset

Load any HuggingFace dataset with `load_dataset`, then pass the split straight into `load_from_hf_dataset()`.

### Parameters at a glance

| Parameter | Purpose |
|---|---|
| `text_fields` | Column(s) whose text becomes the Document content. Pass a list to merge multiple columns (e.g. instruction + output). |
| `meta_fields` | Columns to keep as metadata (e.g. `"category"`, `"source"`). Everything else is dropped. |
| `concat_sep` | Separator inserted between merged field values. Default `"\n"`. |

In [ ]:
from datasets import load_dataset

# ── Load the dataset from the Hub ──────────────────────────────────────────
dataset = load_dataset("BAAI/IndustryInstruction_Finance-Economics")["train"]
dataset = dataset.select(range(75000))

# Peek at column names so you know what to pass to text_fields / meta_fields
print("Columns:", dataset.column_names)
print("Example row:", dataset[0])

industry_instruction_semantic_cluster_de(…):   0%|          | 0.00/121M [00:00<?, ?B/s]

(…)1%E8%9E%8D_%E7%BB%8F%E6%B5%8E_eval.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/122090 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Columns: ['deita_score', 'rw_score', 'id', 'instruction', 'lang', 'length', 'conversations']
Example row: {'deita_score': 8.7799195449, 'rw_score': -2.078125, 'id': 'generate_from_given_text', 'instruction': '', 'lang': 'en', 'length': 35, 'conversations': [{'from': 'human', 'value': 'What are the main concerns of Henry Meier regarding the emphasis on cooperative principles in the credit union industry, and how does he think they should be balanced with other considerations?'}, {'from': 'gpt', 'value': "Henry Meier's main concern is that the emphasis on cooperative principles may overshadow more critical issues facing credit unions, such as lending and legislative impact. He believes that an appreciation for the impact of legislation on credit unions is more important than the cooperative structure. Meier also thinks that the cooperative structure is not as crucial to potential young members as some might think, and that convenience and good service are more important to them."}]}


In [ ]:
# ── Instantiate ───────────────────────────────────────────────────────────
rag = RAG(
    ollama_model             = "mistral:latest",
    bertopic_embedding_model = "paraphrase-multilingual-MiniLM-L12-v2",
    min_topic_size           = 150,
    top_k                    = 3,
    use_gpu                  = True,
)

# ── ONE-TIME SETUP ────────────────────────────────────────────────────────
raw_docs = rag.load_from_hf_dataset(
    dataset,
    text_fields = ["conversations"],
)
chunks = rag.chunk_documents(raw_docs)
rag.create_vectorstore(chunks)
rag.save_index("faiss_index")   # optional — skip setup on next restart

  ✔ FAISS embedder running on CUDA.


/tmp/ipykernel_2558/1782743409.py:358: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embedder = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

🧠 [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 00:31  stage: UMAP   

2026-04-13 14:00:41,660 - BERTopic - Cluster - Completed ✓
2026-04-13 14:00:41,711 - BERTopic - Representation - Fine-tuning topics using representation models.


🧠 [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 00:38  stage: UMAP   

2026-04-13 14:00:49,229 - BERTopic - Representation - Completed ✓


🧠 [██████████████████████████████] 00:40  ✔ Done!                    
  ✔ Found 113 topics across 228291 sentences.
  → Assembling chunks (vectorized)...
  ✔ Produced 98138 topic-coherent chunks.
  → Building FAISS index from 98138 chunks...
  ✔ FAISS index built with 98138 chunks.
  ✔ FAISS index saved to 'faiss_index'.


In [ ]:
# ── QUERY AS MANY TIMES AS YOU LIKE ──────────────────────────────────────
print(rag.query("What are the key drivers of inflation in emerging markets?"))
# ── NEXT SESSION: skip setup entirely ────────────────────────────────────
# rag.load_index("faiss_index")
# print(rag.query("anything..."))

 The provided context does not directly discuss the key drivers of inflation in emerging markets. However, it does imply that factors such as energy supply issues leading to high energy prices and monetary policy can influence inflation levels. In the case of emerging markets, additional factors like fiscal policies, exchange rates, and trade imbalances may also significantly impact inflation rates.


In [ ]:
import pandas as pd

# 1. Define the complete list of queries
queries = [
    "What are the key drivers of inflation in emerging markets?",
    "Explain quantitative easing.",
    "What is the difference between fiscal and monetary policy?",
    "what best describes blockchain and its' relationship with finance?",
    "How do fluctuating interest rates impact venture capital funding?",
    "What are the commercial implications of integrating generative AI into customer support operations?",
    "How do geopolitical tensions affect global supply chain resilience?",
    "What are the potential economic benefits of using distributed ledger technology for cross-border payments?",
    "What are the main metrics used to calculate customer lifetime value (CLV) in a subscription model?",
    "How does corporate ESG (Environmental, Social, and Governance) reporting influence institutional investor decisions?"
]

# 2. Initialize an empty list to store the data
results_data = []

# 3. Iterate through the queries and collect the LLM's responses
print("Processing queries. This may take a moment depending on the LLM's speed...")
for query in queries:
    # Assuming rag.query() returns the text response from the LLM
    try:
        response = rag.query(query)
    except Exception as e:
        response = f"Error generating response: {e}"

    results_data.append({
        "Question": query,
        "LLM_Result": response
    })

# 4. Create the Pandas DataFrame
df = pd.DataFrame(results_data)

# 5. Display a preview of the DataFrame
print("\nPreview of Results:")
print(df.head())

# 6. Save the DataFrame to a CSV file
output_filename = "business_rag_results.csv"
df.to_csv(output_filename, index=False)
print(f"\nSuccessfully saved all queries and results to {output_filename}")

Processing queries. This may take a moment depending on the LLM's speed...

Preview of Results:
                                            Question  \
0  What are the key drivers of inflation in emerg...   
1                       Explain quantitative easing.   
2  What is the difference between fiscal and mone...   
3  what best describes blockchain and its' relati...   
4  How do fluctuating interest rates impact ventu...   

                                          LLM_Result  
0   The context does not provide specific informa...  
1   Quantitative easing is a monetary policy tool...  
2   The difference between fiscal policy and mone...  
3   Blockchain is a decentralized and transparent...  
4   Fluctuating interest rates impact venture cap...  

Successfully saved all queries and results to business_rag_results.csv


## (Optional) Inspect Topics

After calling `analyze()`, you can introspect the BERTopic model to see which topics were discovered across your corpus.

In [ ]:
# Peek at the top terms per topic discovered during the last analyze() run
topic_info = rag.chunker.topic_model.get_topic_info()
print(topic_info[["Topic", "Count", "Name"]].to_string(index=False))

 Topic  Count                                             Name
    -1 140484                                 -1_the_and_to_of
     0   7310                       0_china_chinese_value_from
     1   6114                  1_china_chinese_global_economic
     2   5663                       2_tax_taxes_income_revenue
     3   3467                      3_stock_stocks_price_market
     4   2983                 4_trade_countries_global_foreign
     5   2914         5_innovation_development_growth_economic
     6   2885 6_regulatory_transparency_regulations_compliance
     7   2650                           7_stock_value_from_gpt
     8   2526                     8_risk_risks_management_风险管理
     9   2084                    9_european_eu_eurozone_greece
    10   1973            10_pension_retirement_security_social
    11   1766                   11_european_eu_eurozone_greece
    12   1643                                12_首先_其次_此外_value
    13   1615                           13_trade_value_

In [ ]:
# Visualise topic clusters interactively (requires a Jupyter environment)
rag.chunker.topic_model.visualize_topics()